# Introduction (Himmelblau's function)



Let's use ``blop`` to minimize Himmelblau's function, which has four global minima:

In [ ]:
from blop.utils import prepare_re_env

%run -i $prepare_re_env.__file__ --db-type=temp

In [ ]:
import Shadow
import numpy as np
import matplotlib as mpl
from matplotlib import pyplot as plt

In [ ]:
# the toroid() function can calculate the corresponding beam size for a given (x_rot, y_rot) pair. Shadow is used for beamline simulation.
def toroid(X_ROT,Y_ROT):
    beam = Shadow.Beam()
    oe0 = Shadow.Source()
    oe1 = Shadow.OE()
    oe2 = Shadow.OE()

    oe1.X_ROT = X_ROT
    oe1.Y_ROT = Y_ROT

    

    oe0.FDISTR = 3
    oe0.F_PHOT = 0
    oe0.HDIV1 = 0.0
    oe0.HDIV2 = 0.0
    oe0.IDO_VX = 0
    oe0.IDO_VZ = 0
    oe0.IDO_X_S = 0
    oe0.IDO_Y_S = 0
    oe0.IDO_Z_S = 0
    oe0.ISTAR1 = 5676561
    oe0.PH1 = 1000.0
    oe0.SIGDIX = 5e-05
    oe0.SIGDIZ = 5e-05
    oe0.SIGMAX = 1e-05
    oe0.SIGMAZ = 1e-05
    oe0.VDIV1 = 0.0
    oe0.VDIV2 = 0.0
    oe1.DUMMY = 100.0
    oe1.FMIRR = 3
    oe1.FWRITE = 1
    oe1.F_EXT = 1
    oe1.F_MOVE = 1
    oe1.R_MAJ = 305.3065
    oe1.R_MIN = 0.655
    oe1.T_IMAGE = 0.0

    oe2.ALPHA = 90.0
    oe2.DUMMY = 100.0
    oe2.FCYL = 1
    oe2.FMIRR = 2
    oe2.FWRITE = 1
    oe2.F_DEFAULT = 0
    oe2.SIMAG = 5.0
    oe2.SSOUR = 1000000.0
    oe2.THETA = 88.0
    oe2.T_IMAGE = 5.0
    oe2.T_SOURCE = 5.0

    beam.genSource(oe0)

    beam.traceOE(oe1,1)

    beam.traceOE(oe2,2)

    x_fhwm_m = beam.histo1(1)['fwhm']
    y_fwhm_m = beam.histo1(3)['fwhm']

    a=x_fhwm_m*1e6
    b=y_fwhm_m*1e6
    size_um=np.sqrt(a**2+b**2)
   
    return size_um

There are several things that our agent will need. The first ingredient is some degrees of freedom (these are always `ophyd` devices) which the agent will move around to different inputs within each DOF's bounds (the second ingredient). We define these here:

In [ ]:
from blop import DOF

dofs = [
    DOF(name="x_rot", search_bounds=(-0.2, 0.2)),
    DOF(name="y_rot", search_bounds=(-0.2, 0.2)),
]

We also need to give the agent something to do. We want our agent to look in the feedback for a variable called 'beamsize', and try to minimize it.

In [ ]:
from blop import Objective

objectives = [Objective(name="beamsize", description="toroid's function", target="min")]

In our digestion function, we define our objective as a deterministic function of the inputs:

In [ ]:
def digestion(db, uid):
    products = db[uid].table()

    for index, entry in products.iterrows():
        products.loc[index, "beamsize"] = toroid(entry.x_rot, entry.y_rot)
       

    return products

We then combine these ingredients into an agent, giving it an instance of ``databroker`` so that it can see the output of the plans it runs.

In [ ]:
from blop import Agent

agent = Agent(
    dofs=dofs,
    objectives=objectives,
    digestion=digestion,
    db=db,
)

Without any data, we can't make any inferences about what the function looks like, and so we can't use any non-trivial acquisition functions. Let's start by quasi-randomly sampling the parameter space, and plotting our model of the function:

In [ ]:
RE(agent.learn("quasi-random", n=32))
agent.plot_objectives()

To decide which points to sample, the agent needs an acquisition function. The available acquisition function are here:

In [ ]:
agent.all_acq_funcs

Now we can start to learn intelligently. Using the shorthand acquisition functions shown above, we can see the output of a few different ones:

In [ ]:
agent.plot_acquisition(acq_func="qei")

To decide where to go, the agent will find the inputs that maximize a given acquisition function:

In [ ]:
agent.ask("qei", n=1)

We can also ask the agent for multiple points to sample and it will jointly maximize the acquisition function over all sets of inputs, and find the most efficient route between them:

In [ ]:
res = agent.ask("qei", n=8, route=True)
agent.plot_acquisition(acq_func="qei")
plt.scatter(*res["points"].T, marker="d", facecolor="w", edgecolor="k")
plt.plot(
    *res["points"].T,
    color="r",
)

All of this is automated inside the ``learn`` method, which will find a point (or points) to sample, sample them, and retrain the model and its hyperparameters with the new data. To do 4 learning iterations of 8 points each, we can run

In [ ]:
RE(agent.learn("qei", n=4, iterations=8))

Our agent has found all the global minima of the defined function toroid using Bayesian optimization, and we can ask it for the best point: 

In [ ]:
agent.plot_objectives()
print(agent.best)